In [1]:
import tkinter as tk
import time
import copy

# Danh sách 11 huyện/thị/thành của Tiền Giang
DISTRICTS = [
    'Cái Bè', 'Huyện Cai Lậy', 'TX Cai Lậy', 'Tân Phước', 'Châu Thành',
    'Mỹ Tho', 'Chợ Gạo', 'Gò Công Tây', 'TX Gò Công', 'Gò Công Đông', 'Tân Phú Đông'
]

# Ràng buộc kề nhau (Dựa trên bản đồ hành chính Tiền Giang)
NEIGHBORS = {
    'Cái Bè': ['Huyện Cai Lậy'],
    'Huyện Cai Lậy': ['Cái Bè', 'TX Cai Lậy', 'Tân Phước', 'Châu Thành'],
    'TX Cai Lậy': ['Huyện Cai Lậy', 'Tân Phước', 'Châu Thành'],
    'Tân Phước': ['Huyện Cai Lậy', 'TX Cai Lậy', 'Châu Thành'],
    'Châu Thành': ['Huyện Cai Lậy', 'TX Cai Lậy', 'Tân Phước', 'Mỹ Tho', 'Chợ Gạo'],
    'Mỹ Tho': ['Châu Thành', 'Chợ Gạo'],
    'Chợ Gạo': ['Châu Thành', 'Mỹ Tho', 'Gò Công Tây', 'Tân Phú Đông'],
    'Gò Công Tây': ['Chợ Gạo', 'TX Gò Công', 'Tân Phú Đông'],
    'TX Gò Công': ['Gò Công Tây', 'Gò Công Đông', 'Tân Phú Đông'],
    'Gò Công Đông': ['TX Gò Công', 'Tân Phú Đông'],
    'Tân Phú Đông': ['Chợ Gạo', 'Gò Công Tây', 'TX Gò Công', 'Gò Công Đông']
}

# Đảm bảo tính đối xứng của đồ thị (A kề B thì B kề A)
for u in list(NEIGHBORS.keys()):
    for v in NEIGHBORS[u]:
        if v not in NEIGHBORS: NEIGHBORS[v] = []
        if u not in NEIGHBORS[v]: NEIGHBORS[v].append(u)

# Tọa độ tương đối trên Canvas (x, y) để hiển thị sơ đồ Tiền Giang
COORDS = {
    'Cái Bè': (80, 350),
    'Huyện Cai Lậy': (190, 400),
    'TX Cai Lậy': (240, 280),
    'Tân Phước': (320, 150),
    'Châu Thành': (410, 350),
    'Mỹ Tho': (450, 460),
    'Chợ Gạo': (560, 400),
    'Gò Công Tây': (660, 450),
    'TX Gò Công': (750, 350),
    'Gò Công Đông': (840, 400),
    'Tân Phú Đông': (750, 560)
}

COLOR_NAMES = ['Đỏ', 'Xanh lá', 'Xanh dương', 'Vàng']
COLOR_CODES = {'Đỏ': '#ff4d4d', 'Xanh lá': '#4dff4d', 'Xanh dương': '#4da6ff', 'Vàng': '#ffff4d', 'Trắng': '#ffffff'}

class MapColoringApp:
    def __init__(self, root):
        self.root = root
        self.root.title("CSP - Map Coloring Tiền Giang")
        self.root.geometry("1400x900")

        self.left_frame = tk.Frame(root, bg='#f0f0f0')
        self.left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        self.right_frame = tk.Frame(root, width=500, bg='white')
        self.right_frame.pack(side=tk.RIGHT, fill=tk.Y)
        self.right_frame.pack_propagate(False)

        # Đặt frame nút bấm ở dưới cùng, pad thêm để không đụng lề dưới
        self.btn_frame = tk.Frame(self.left_frame, bg='#f0f0f0')
        self.btn_frame.pack(side=tk.BOTTOM, pady=20)
        
        self.btn_bt = tk.Button(self.btn_frame, text="Backtracking Thuần", command=lambda: self.start_csp(False), font=("Arial", 12, "bold"), bg="#2196F3", fg="white", width=20, cursor="hand2")
        self.btn_bt.pack(side=tk.LEFT, padx=20)
        
        self.btn_fc = tk.Button(self.btn_frame, text="Backtracking + Forward Checking", command=lambda: self.start_csp(True), font=("Arial", 12, "bold"), bg="#4CAF50", fg="white", width=30, cursor="hand2")
        self.btn_fc.pack(side=tk.LEFT, padx=20)

        # Canvas hiển thị bản đồ chiếm toàn bộ phần trên
        self.canvas = tk.Canvas(self.left_frame, bg='#e3f2fd')
        self.canvas.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        tk.Label(self.right_frame, text="Log thực thi thuật toán", font=("Arial", 14, "bold"), bg='white').pack(pady=5)
        self.log_text = tk.Text(self.right_frame, font=("Consolas", 10), wrap=tk.WORD, padx=5, pady=5, bg="#f8f9fa")
        self.scrollbar = tk.Scrollbar(self.right_frame, command=self.log_text.yview)
        self.log_text.configure(yscrollcommand=self.scrollbar.set)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        self.node_shapes = {}
        self.draw_map()

    def draw_map(self):
        drawn_edges = set()
        for u in DISTRICTS:
            for v in NEIGHBORS[u]:
                edge = tuple(sorted((u, v)))
                if edge not in drawn_edges:
                    x1, y1 = COORDS[u]
                    x2, y2 = COORDS[v]
                    # Cải thiện đường nối: Nét liền (solid), màu sậm hơn, độ dày=3 dễ nhìn
                    self.canvas.create_line(x1, y1, x2, y2, fill="#555555", width=3)
                    drawn_edges.add(edge)

        for node in DISTRICTS:
            x, y = COORDS[node]
            w, h = 50, 22
            
            self.canvas.create_rectangle(x-w+3, y-h+3, x+w+3, y+h+3, fill='#9e9e9e', outline='')
            rect = self.canvas.create_rectangle(x-w, y-h, x+w, y+h, fill='white', outline='#212121', width=2)
            self.canvas.create_text(x, y, text=node, font=("Arial", 9, "bold"), fill="#212121")
            self.node_shapes[node] = rect

    def write_log(self, text):
        self.log_text.insert(tk.END, text + "\n")
        self.log_text.see(tk.END)
        self.root.update()

    def update_node_color(self, node, color_name):
        hex_color = COLOR_CODES.get(color_name, "white")
        self.canvas.itemconfig(self.node_shapes[node], fill=hex_color)
        self.root.update()
        time.sleep(0.4) 

    def format_assignment(self, assignment):
        return "{" + ", ".join([f"{k}={v}" for k, v in assignment.items()]) + "}"

    def print_initial_info(self):
        self.write_log("=== ĐỊNH NGHĨA BÀI TOÁN CSP ===")
        self.write_log(f"- Biến: {{{', '.join(DISTRICTS)}}}")
        self.write_log(f"- Miền giá trị: {{{', '.join(COLOR_NAMES)}}}")
        
        constraints = []
        drawn = set()
        for u in DISTRICTS:
            for v in NEIGHBORS[u]:
                edge = tuple(sorted((u, v)))
                if edge not in drawn:
                    constraints.append(f"{u}≠{v}")
                    drawn.add(edge)
        self.write_log(f"- Ràng buộc: {', '.join(constraints)}")
        self.write_log("===============================\n")

    def start_csp(self, use_fc):
        self.btn_bt.config(state=tk.DISABLED)
        self.btn_fc.config(state=tk.DISABLED)
        self.log_text.delete(1.0, tk.END)
        
        for node in DISTRICTS:
            self.update_node_color(node, 'Trắng')

        self.print_initial_info()
        algo_name = "Backtracking + Forward Checking" if use_fc else "Backtracking Thuần"
        self.write_log(f"=== KHỞI CHẠY: {algo_name} ===\n")

        assignment = {}
        domains = {node: list(COLOR_NAMES) for node in DISTRICTS}
        
        self.step_counter = 1
        self.write_log(f"Bước {self.step_counter}: Bắt đầu phép gán rỗng. Assignment={{}}")
        self.step_counter += 1
        
        success = self.backtrack(assignment, domains, use_fc)
        
        if success:
            self.write_log("\n=> HOÀN THÀNH TÔ MÀU BẢN ĐỒ TẤT CẢ RÀNG BUỘC!")
        else:
            self.write_log("\n=> KHÔNG TÌM THẤY GIẢI PHÁP!")
            
        self.btn_bt.config(state=tk.NORMAL)
        self.btn_fc.config(state=tk.NORMAL)

    def backtrack(self, assignment, domains, use_fc):
        if len(assignment) == len(DISTRICTS):
            return True

        unassigned_vars = [v for v in DISTRICTS if v not in assignment]
        var = unassigned_vars[0] 
        
        self.write_log(f"\nBước {self.step_counter}: Chọn biến {var}")
        self.step_counter += 1

        for color in domains[var]:
            self.write_log(f"  + Thử gán: {var} = {color}")
            
            is_valid = True
            violated_neighbor = None
            
            for neighbor in NEIGHBORS[var]:
                if neighbor in assignment and assignment[neighbor] == color:
                    is_valid = False
                    violated_neighbor = neighbor
                    break
            
            if not is_valid:
                self.write_log(f"  -> Không hợp lệ (Vi phạm ràng buộc: {var} ≠ {violated_neighbor} đang cùng màu {color})")
                continue 
                
            self.write_log(f"  -> Hợp lệ")
            assignment[var] = color
            self.update_node_color(var, color)
            self.write_log(f"  -> Assignment = {self.format_assignment(assignment)}")

            if use_fc:
                new_domains = copy.deepcopy(domains)
                new_domains[var] = [color]
                
                fc_success = True
                updated_domains = []
                empty_domain_var = None

                for neighbor in NEIGHBORS[var]:
                    if neighbor not in assignment:
                        if color in new_domains[neighbor]:
                            new_domains[neighbor].remove(color)
                            updated_domains.append(neighbor)
                            if len(new_domains[neighbor]) == 0:
                                fc_success = False
                                empty_domain_var = neighbor

                if updated_domains:
                    self.write_log("  -> Update domain các biến kề (Forward Checking):")
                    for neighbor in updated_domains:
                        domain_str = "{" + ", ".join(new_domains[neighbor]) + "}"
                        self.write_log(f"     * Miền giá trị của {neighbor} = {domain_str}")

                if fc_success:
                    result = self.backtrack(assignment, new_domains, use_fc)
                    if result:
                        return True
                else:
                    self.write_log(f"  => CẢNH BÁO: Miền giá trị của {empty_domain_var} bị rỗng (Forward Checking)!")
            else:
                result = self.backtrack(assignment, domains, use_fc)
                if result:
                    return True

            del assignment[var]
            self.update_node_color(var, 'Trắng')
            self.write_log(f"\nBước {self.step_counter}: QUAY LUI (Backtrack)!")
            self.write_log(f"  + Hủy gán {var}={color}, loại bỏ hướng đi này.")
            self.step_counter += 1

        return False

# Khởi chạy ứng dụng
if __name__ == '__main__':
    root = tk.Tk()
    app = MapColoringApp(root)
    root.mainloop()
